In [ ]:
df = pd.read_csv("../data/processed/cobertura_vacinal_por_uf.csv")
df_brasil = pd.read_csv("../data/processed/cobertura_vacinal_brasil.csv")

In [ ]:
dados_2022 = df[df["ano"] == 2022].groupby("uf")["cobertura"].mean().reset_index()

fig = px.choropleth(
    dados_2022,
    geojson=geojson_brasil,
    locations="uf",
    featureidkey="properties.name",
    color="cobertura",
    color_continuous_scale="RdYlGn",
    range_color=(60, 95),
    scope="south america",
    title="Cobertura Vacinal Média por Estado (2022)"
)
fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig.show()
fig.write_image("../dashboard/mapa_cobertura_2022.png")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("../data/processed/cobertura_vacinal_por_uf.csv")
df_brasil = pd.read_csv("../data/processed/cobertura_vacinal_brasil.csv")

In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(data=df_brasil, x="ano", y="cobertura", hue="vacina", marker="o")
plt.axhline(y=95, color="gray", linestyle="--", alpha=0.5, label="Meta ideal (95%)")
plt.axvspan(2020, 2021, alpha=0.1, color="red", label="Período pandemia")
plt.title("Evolução da Cobertura Vacinal no Brasil (2013-2022)")
plt.xlabel("Ano")
plt.ylabel("Cobertura Vacinal (%)")
plt.legend(title="Vacina", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../dashboard/grafico_evolucao_temporal.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
comparacao = (
    df[df["ano"].isin([2019, 2022])]
    .groupby(["uf", "ano"])["cobertura"]
    .mean()
    .reset_index()
    .pivot(index="uf", columns="ano", values="cobertura")
)
comparacao["variacao"] = comparacao[2022] - comparacao[2019]
comparacao = comparacao.sort_values("variacao")

plt.figure(figsize=(10, 10))
cores = ["firebrick" if x < 0 else "seagreen" for x in comparacao["variacao"]]
plt.barh(comparacao.index, comparacao["variacao"], color=cores)
plt.axvline(x=0, color="black", linewidth=0.8)
plt.title("Variação da Cobertura Vacinal Média por Estado (2019 → 2022)")
plt.xlabel("Variação em pontos percentuais")
plt.ylabel("Estado")
plt.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig("../dashboard/grafico_ranking_estados.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import plotly.express as px
import json
import urllib.request

url_estados = "https://raw.githubusercontent.com/codeforgermany/click_that_hood/main/public/data/brazil-states.geojson"

with urllib.request.urlopen(url_estados) as response:
    geojson_brasil = json.loads(response.read())

dados_2022 = df[df["ano"] == 2022].groupby("uf")["cobertura"].mean().reset_index()

fig = px.choropleth(
    dados_2022,
    geojson=geojson_brasil,
    locations="uf",
    featureidkey="properties.name",
    color="cobertura",
    color_continuous_scale="RdYlGn",
    range_color=(50, 110),
    scope="south america",
    title="Cobertura Vacinal Média por Estado (2022)"
)
fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig.show()
fig.write_image("../dashboard/mapa_cobertura_2022.png")

In [ ]:
queda_nacional = (
    df_brasil[df_brasil["ano"].isin([2019, 2022])]
    .pivot(index="vacina", columns="ano", values="cobertura")
)
queda_nacional["variacao"] = queda_nacional[2022] - queda_nacional[2019]
print("=== Variação nacional 2019 -> 2022 ===")
print(queda_nacional)

print("\n=== 3 estados que mais perderam cobertura ===")
print(comparacao.head(3))

print("\n=== 3 estados com melhor desempenho ===")
print(comparacao.tail(3))

pior_2022 = df[df["ano"] == 2022].groupby("uf")["cobertura"].mean().sort_values().head(3)
print("\n=== 3 estados com menor cobertura em 2022 ===")
print(pior_2022)